# Massive SPX/VIX/SPY Database — Starter

This notebook validates access, downloads underlyings, builds Parquet and DuckDB storage, discovers SPX option identifiers, saves option metadata, collects current option-chain snapshots, and demonstrates a controlled historical option sample.

**The API key is loaded from `.env` and is never displayed.**

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import os
import sys
from datetime import date, timedelta

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("Run this notebook from the massive_market_database folder.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from massive_database import (
    MassiveREST,
    data_quality_report,
    discover_option_underlying,
    download_aggregate_history,
    download_selected_option_bars,
    initialize_duckdb,
    latest_regular_session_price,
    normalize_aggregates,
    normalize_option_chain,
    normalize_option_contracts,
    register_parquet_views,
    select_near_atm_contracts,
    write_snapshot_parquet,
)

DATA_ROOT = PROJECT_ROOT / "data"
DB_PATH = DATA_ROOT / "market.duckdb"
DATA_ROOT.mkdir(exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.getenv("MASSIVE_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "MASSIVE_API_KEY is missing. Copy .env.example to .env, add your key, then rerun."
    )

client = MassiveREST(API_KEY)
con = initialize_duckdb(DB_PATH)
print("Configuration loaded. The API key is present but hidden.")
print("DuckDB:", DB_PATH)

## 1. Validate underlying access

This searches recent weekdays rather than assuming yesterday was a trading session.

In [ ]:
def recent_data_sample(ticker: str, asset_class: str, lookback_days: int = 14):
    today = date.today()
    for days_back in range(1, lookback_days + 1):
        candidate = (today - timedelta(days=days_back)).isoformat()
        try:
            raw = client.aggregates(ticker, candidate, candidate)
            if raw:
                return candidate, normalize_aggregates(raw, ticker, asset_class)
        except Exception as exc:
            return None, pd.DataFrame({"ticker": [ticker], "error": [str(exc)]})
    return None, pd.DataFrame()

checks, samples = [], {}
for ticker, asset_class in [("SPY", "stock"), ("I:SPX", "index"), ("I:VIX", "index")]:
    sample_date, frame = recent_data_sample(ticker, asset_class)
    samples[ticker] = frame
    checks.append({
        "ticker": ticker,
        "asset_class": asset_class,
        "sample_date": sample_date,
        "rows": len(frame),
        "status": "ok" if sample_date else "no data or no access",
    })

access_report = pd.DataFrame(checks)
display(access_report)

In [ ]:
spx_sample = samples.get("I:SPX", pd.DataFrame())
display(spx_sample.head())
data_quality_report(spx_sample)

## 2. Proof-of-concept storage

The downloader is incremental. Successfully downloaded sessions are skipped on reruns.

In [ ]:
end_date = (date.today() - timedelta(days=1)).isoformat()
start_date = (date.today() - timedelta(days=10)).isoformat()

for ticker, asset_class in [("SPY", "stock"), ("I:SPX", "index"), ("I:VIX", "index")]:
    result = download_aggregate_history(
        client=client,
        con=con,
        data_root=DATA_ROOT,
        ticker=ticker,
        asset_class=asset_class,
        start_date=start_date,
        end_date=end_date,
    )
    print(ticker)
    display(pd.DataFrame([r.__dict__ for r in result]).tail())

print("Views created:", register_parquet_views(con, DATA_ROOT))

In [ ]:
display(
    con.execute(
        """
        SELECT ticker, min(timestamp_et) AS first_bar,
               max(timestamp_et) AS last_bar, count(*) AS rows
        FROM underlying_minute_1
        GROUP BY ticker
        ORDER BY ticker
        """
    ).df()
)

display(
    con.execute(
        """
        SELECT ticker, timestamp_et, open, high, low, close, volume
        FROM underlying_15min
        ORDER BY timestamp_utc DESC
        LIMIT 12
        """
    ).df()
)

## 3. Full underlying history

The default range is two years. Move the start date earlier only where plan history allows.

In [ ]:
RUN_FULL_UNDERLYING_DOWNLOAD = False
FULL_START_DATE = (
    pd.Timestamp.today().normalize() - pd.DateOffset(years=2)
).date().isoformat()
FULL_END_DATE = (date.today() - timedelta(days=1)).isoformat()

if RUN_FULL_UNDERLYING_DOWNLOAD:
    summaries = []
    for ticker, asset_class in [("SPY", "stock"), ("I:SPX", "index"), ("I:VIX", "index")]:
        results = download_aggregate_history(
            client=client,
            con=con,
            data_root=DATA_ROOT,
            ticker=ticker,
            asset_class=asset_class,
            start_date=FULL_START_DATE,
            end_date=FULL_END_DATE,
        )
        summaries.extend(r.__dict__ for r in results)

    register_parquet_views(con, DATA_ROOT)
    display(
        pd.DataFrame(summaries)
        .groupby(["ticker", "status"])
        .size()
        .unstack(fill_value=0)
    )
else:
    print("Disabled. Set RUN_FULL_UNDERLYING_DOWNLOAD = True when ready.")
    print(FULL_START_DATE, "to", FULL_END_DATE)

## 4. Discover and save SPX option contracts

The notebook tests both `I:SPX` and `SPX` instead of assuming the option-underlying identifier.

In [ ]:
reference_dates = access_report.loc[
    access_report["ticker"].eq("I:SPX"), "sample_date"
].dropna()
reference_date = reference_dates.iloc[0] if len(reference_dates) else FULL_END_DATE

option_underlying, discovery_report = discover_option_underlying(
    client,
    candidates=("I:SPX", "SPX"),
    as_of=reference_date,
)
display(discovery_report)
print("Selected option underlying:", option_underlying)

In [ ]:
if option_underlying:
    expiration_lte = (
        pd.Timestamp(reference_date) + pd.Timedelta(days=14)
    ).date().isoformat()

    contracts_raw = client.option_contracts(
        underlying_ticker=option_underlying,
        as_of=reference_date,
        expiration_date_gte=reference_date,
        expiration_date_lte=expiration_lte,
        expired=True,
        limit=1000,
    )
    contracts = normalize_option_contracts(contracts_raw, reference_date)
    display(contracts.head())
    print("Contracts:", len(contracts))

    if not contracts.empty:
        path = write_snapshot_parquet(
            contracts,
            DATA_ROOT,
            "option_contracts",
            reference_date,
            "contracts.parquet",
        )
        print("Saved:", path)
        register_parquet_views(con, DATA_ROOT)
else:
    contracts = pd.DataFrame()
    print("No option underlying was discovered. Review the access report.")

## 5. Current option-chain snapshot

This is for collection from now onward. It does not backfill historical Greeks or open interest.

In [ ]:
COLLECT_CURRENT_CHAIN = False

if COLLECT_CURRENT_CHAIN and option_underlying:
    today = date.today().isoformat()
    expiry_end = (
        pd.Timestamp(today) + pd.Timedelta(days=14)
    ).date().isoformat()

    chain_raw = client.option_chain_snapshot(
        underlying_asset=option_underlying,
        expiration_date_gte=today,
        expiration_date_lte=expiry_end,
        limit=250,
    )
    chain = normalize_option_chain(chain_raw)
    display(chain.head())
    print("Snapshot rows:", len(chain))

    if not chain.empty:
        path = write_snapshot_parquet(
            chain,
            DATA_ROOT,
            "option_chain_snapshots",
            today,
            "spx_chain.parquet",
        )
        print("Saved:", path)
        register_parquet_views(con, DATA_ROOT)
else:
    print("Disabled. Set COLLECT_CURRENT_CHAIN = True after checking plan access.")

## 6. Controlled historical option-price sample

This uses the latest SPX price at or before 15:00 ET, limits the universe to ±5% of spot, selects no more than 20 contracts, and downloads one session.

In [ ]:
RUN_OPTION_SAMPLE = False
register_parquet_views(con, DATA_ROOT)

if RUN_OPTION_SAMPLE and option_underlying and not contracts.empty:
    spot = latest_regular_session_price(
        con,
        "I:SPX",
        reference_date,
        "15:00:00",
    )
    print("SPX reference price:", spot)

    if spot is None:
        raise RuntimeError("No SPX price found for the reference session.")

    selected = select_near_atm_contracts(
        contracts,
        spot,
        max_contracts=20,
        moneyness_pct=0.05,
    )
    display(
        selected[
            [
                "ticker",
                "contract_type",
                "expiration_date",
                "strike_price",
                "distance_to_spot",
            ]
        ]
    )

    report = download_selected_option_bars(
        client,
        con,
        DATA_ROOT,
        selected,
        reference_date,
    )
    display(report)
    register_parquet_views(con, DATA_ROOT)
else:
    print("Disabled. Set RUN_OPTION_SAMPLE = True after reviewing the contract universe.")

## 7. Audit log

Errors and empty sessions remain visible rather than being silently discarded.

In [ ]:
display(
    con.execute(
        """
        SELECT dataset, ticker, status,
               count(*) AS sessions, sum(rows) AS rows
        FROM ingestion_log
        GROUP BY dataset, ticker, status
        ORDER BY dataset, ticker, status
        """
    ).df()
)

display(
    con.execute(
        """
        SELECT *
        FROM ingestion_log
        WHERE status = 'error'
        ORDER BY ingested_at_utc DESC
        LIMIT 20
        """
    ).df()
)

## Next notebook

After the underlying download is complete, build a canonical session calendar, align SPX/VIX/SPY without look-ahead, freeze the option-universe sampling rule, and create modelling tables separately from raw storage.